# Projeto de Machine Learning – Análise e Predição de Inadimplência em Crédito

## Metodologia: CRISP-DM

Este projeto tem como objetivo analisar e prever a inadimplência de clientes a partir de dados financeiros, utilizando modelos de aprendizado supervisionado.

Além disso, busca-se identificar padrões associados ao risco de crédito, contribuindo para a tomada de decisão em contextos financeiros e de gestão de risco.

O projeto também dialoga com aplicações em detecção de fraudes e segurança financeira, uma vez que ambos os contextos envolvem a identificação de padrões de risco em dados transacionais.

## Contextualização do Dataset

O dataset utilizado neste projeto corresponde ao histórico de empréstimos concedidos pela plataforma **Lending Club**, uma empresa norte-americana de crédito peer-to-peer (P2P), que conecta investidores a pessoas físicas interessadas em obter financiamento.

Os dados abrangem operações realizadas entre **2007 e 2018**, contendo informações detalhadas sobre os clientes, características dos empréstimos e o status final de pagamento.

### Features selecionadas:
| Feature | Descrição |
|---------|-----------|
| loan_amnt | Valor do empréstimo solicitado |
| term | Prazo do empréstimo (36 ou 60 meses) |
| int_rate | Taxa de juros |
| installment | Valor da parcela mensal |
| annual_inc | Renda anual do cliente |
| dti | Índice de endividamento |
| fico_range_high | Score de crédito máximo |
| revol_util | Utilização do crédito rotativo |
| loan_status | Status do empréstimo |

> **Nota:** Foram selecionadas apenas features disponíveis no momento da concessão do empréstimo, evitando **data leakage**.

# CRISP-DM – Fase 1: Entendimento do Negócio

O objetivo é prever a inadimplência de clientes em operações de crédito.

A variável alvo definida para o modelo é:
- **0** → Cliente adimplente (pagou o empréstimo)
- **1** → Cliente inadimplente (não pagou — Charged Off)

In [ ]:
# CRISP-DM: Fase 2 - Entendimento dos Dados
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, f1_score, recall_score, roc_auc_score, roc_curve
)
from imblearn.over_sampling import SMOTE
import pickle

print("Bibliotecas carregadas ✔")

In [ ]:
# Carregar dataset completo para inspeção inicial
dados = pd.read_csv(
    '/kaggle/input/datasets/wordsforthewise/lending-club/accepted_2007_to_2018Q4.csv.gz',
    compression='gzip',
    low_memory=False
)
print(f"Shape original: {dados.shape}")
dados.head()

In [ ]:
print(dados.info())
print(dados.describe())

In [ ]:
print("Distribuição loan_status:")
print(dados['loan_status'].value_counts())

## CRISP-DM Fase 3 — Preparação dos Dados

### Decisões tomadas:
1. **Amostragem:** 200k registros para otimizar o processamento mantendo representatividade
2. **Filtro loan_status:** mantidos apenas `Fully Paid` e `Charged Off` (resultados finais conhecidos)
3. **Definição do target:** `Charged Off` = inadimplência. `Default` removido por baixa representatividade
4. **Seleção de features:** 8 variáveis disponíveis no momento da concessão (sem data leakage)
5. **Remoção de nulos:** ~3,6% dos dados removidos (proporção baixa, não justifica imputação)
6. **Tratamento de outliers:** Winsorização via IQR para evitar distorções no modelo

In [ ]:
# Amostragem de 200k registros
dados = dados.sample(200000, random_state=42)
print(f"Shape após amostragem: {dados.shape}")

# Filtro apenas status definitivos
dados = dados[dados['loan_status'].isin(['Fully Paid', 'Charged Off'])]
print(f"Shape após filtro: {dados.shape}")
print(dados['loan_status'].value_counts())

In [ ]:
# Selecionar features relevantes
colunas = [
    'loan_amnt', 'term', 'int_rate', 'installment',
    'annual_inc', 'dti', 'fico_range_high', 'revol_util',
    'loan_status'
]
dados = dados[colunas]

# Criar variável target
dados['inadimplente'] = (dados['loan_status'] == 'Charged Off').astype(int)

# Renomear colunas
rename_dict = {
    'loan_amnt': 'valor_emprestimo',
    'term': 'prazo',
    'int_rate': 'taxa_juros',
    'installment': 'parcela',
    'annual_inc': 'renda_anual',
    'dti': 'indice_endividamento',
    'fico_range_high': 'score_credito',
    'revol_util': 'uso_credito_rotativo'
}
dados = dados.rename(columns=rename_dict)

# Converter prazo
dados['prazo'] = dados['prazo'].str.replace(' months', '').astype(int)

print(dados.head())

In [ ]:
# Verificar e remover nulos
print("Nulos por coluna:")
print(dados.isnull().sum())

total_antes = len(dados)  # capturado antes do dropna
print(f"\nTotal antes: {total_antes}")

dados = dados.dropna()
dados = dados.drop(columns=['loan_status'])

total_depois = len(dados)
linhas_removidas = total_antes - total_depois
percentual_removido = (linhas_removidas / total_antes) * 100

print(f"Total após dropna: {total_depois}")
print(f"Linhas removidas: {linhas_removidas} ({percentual_removido:.2f}%)")

In [ ]:
# Tratamento de outliers via IQR (winsorização)
colunas_iqr = ['renda_anual', 'indice_endividamento']

for col in colunas_iqr:
    Q1 = dados[col].quantile(0.25)
    Q3 = dados[col].quantile(0.75)
    IQR = Q3 - Q1
    dados[col] = dados[col].clip(lower=Q1 - 1.5 * IQR, upper=Q3 + 1.5 * IQR)
    print(f"{col}: max após clip = {dados[col].max():.2f}")

## CRISP-DM Fase 3 — Análise Exploratória

In [ ]:
# Distribuição do target
sns.countplot(x='inadimplente', data=dados)
plt.title("Distribuição de Inadimplência")
plt.xlabel("0 = Adimplente | 1 = Inadimplente")
plt.show()

vc = dados['inadimplente'].value_counts()
print(vc)
print(f"\nDesbalanceamento: {vc[0]/vc[1]:.1f}x mais adimplentes")

In [ ]:
# Função de boxplot sem outliers (só para visualização)
def plot_boxplot_sem_outliers(df, col, target='inadimplente', titulo=None):
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    filtrado = df[
        (df[col] >= Q1 - 1.5 * IQR) &
        (df[col] <= Q3 + 1.5 * IQR)
    ]
    sns.boxplot(x=target, y=col, data=filtrado)
    plt.title(titulo or f"{col} vs Inadimplência")
    plt.xlabel("0 = Adimplente | 1 = Inadimplente")
    plt.show()

plot_boxplot_sem_outliers(dados, 'taxa_juros',           titulo='Taxa de Juros vs Inadimplência')
plot_boxplot_sem_outliers(dados, 'score_credito',        titulo='Score de Crédito vs Inadimplência')
plot_boxplot_sem_outliers(dados, 'renda_anual',          titulo='Renda Anual vs Inadimplência')
plot_boxplot_sem_outliers(dados, 'indice_endividamento', titulo='Índice de Endividamento vs Inadimplência')
plot_boxplot_sem_outliers(dados, 'uso_credito_rotativo', titulo='Uso Crédito Rotativo vs Inadimplência')
plot_boxplot_sem_outliers(dados, 'valor_emprestimo',     titulo='Valor do Empréstimo vs Inadimplência')

In [ ]:
# Matriz de correlação
plt.figure(figsize=(10, 8))
sns.heatmap(dados.corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Matriz de Correlação entre Features')
plt.tight_layout()
plt.show()

print("\nCorrelação com inadimplente (ordenado):")
print(dados.corr()['inadimplente'].sort_values(ascending=False))

In [ ]:
# Remover parcela: multicolinearidade alta com valor_emprestimo (correlação ~0.95)
dados = dados.drop(columns=['parcela'])
print(f"Colunas após remoção: {dados.columns.tolist()}")
print(f"Shape final: {dados.shape}")

In [ ]:
# Split treino / teste / validação (70/15/15)
X = dados.drop(columns=['inadimplente'])
y = dados['inadimplente']

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y, shuffle=True
)
X_test, X_val, y_test, y_val = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print(f"Treino:    {X_train.shape[0]} registros ({X_train.shape[0]/len(dados)*100:.1f}%)")
print(f"Teste:     {X_test.shape[0]} registros ({X_test.shape[0]/len(dados)*100:.1f}%)")
print(f"Validação: {X_val.shape[0]} registros ({X_val.shape[0]/len(dados)*100:.1f}%)")

## Experimento 1 — Sem SMOTE (baseline)

Treinamento sem balanceamento de classes para estabelecer uma linha de base.

In [ ]:
print("EXPERIMENTO 1 - Sem SMOTE (baseline)")

listaAlgoritmos_exp1 = [
    RandomForestClassifier(n_estimators=100, random_state=42),
    Pipeline([
        ('scaler', StandardScaler()),
        ('modelo', LogisticRegression(max_iter=3000, random_state=42))
    ]),
    GaussianNB(),
    GradientBoostingClassifier(n_estimators=100, random_state=42),
    MLPClassifier(hidden_layer_sizes=(15,), max_iter=1000, random_state=42)
]

listaModelos_exp1 = []
for algoritmo in listaAlgoritmos_exp1:
    algoritmo.fit(X_train, y_train)
    listaModelos_exp1.append(algoritmo)
    print(f"{algoritmo.__class__.__name__} → treinado ✔")

In [ ]:
resultados_exp1 = []

for modelo in listaModelos_exp1:
    y_pred = modelo.predict(X_test)
    acc  = accuracy_score(y_test, y_pred)
    f1   = f1_score(y_test, y_pred)
    rec  = recall_score(y_test, y_pred)
    resultados_exp1.append({'modelo': modelo.__class__.__name__, 'acc': acc, 'f1': f1, 'recall': rec})

    print(f"\n{'='*50}")
    print(f"  {modelo.__class__.__name__} (Exp 1)")
    print(f"{'='*50}")
    print(classification_report(y_test, y_pred, target_names=['Adimplente', 'Inadimplente']))

    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Adimplente','Inadimplente'],
                yticklabels=['Adimplente','Inadimplente'])
    plt.title(f'Matriz de Confusão — {modelo.__class__.__name__} (Exp 1)')
    plt.ylabel('Real'); plt.xlabel('Previsto')
    plt.tight_layout(); plt.show()

## Experimento 2 — Com SMOTE

Aplicação de SMOTE (Synthetic Minority Over-sampling Technique) para balancear as classes antes do treinamento.

In [ ]:
print("EXPERIMENTO 2 - Com SMOTE")

sm = SMOTE(random_state=42)
X_train_bal, y_train_bal = sm.fit_resample(X_train, y_train)
print(f"Antes:  {y_train.value_counts().to_dict()}")
print(f"Depois: {pd.Series(y_train_bal).value_counts().to_dict()}")

listaAlgoritmos_exp2 = [
    RandomForestClassifier(n_estimators=100, random_state=42),
    Pipeline([
        ('scaler', StandardScaler()),
        ('modelo', LogisticRegression(max_iter=3000, random_state=42))
    ]),
    GaussianNB(),
    GradientBoostingClassifier(n_estimators=100, random_state=42),
    MLPClassifier(hidden_layer_sizes=(15,), max_iter=1000, random_state=42)
]

listaModelos_exp2 = []
for algoritmo in listaAlgoritmos_exp2:
    algoritmo.fit(X_train_bal, y_train_bal)
    listaModelos_exp2.append(algoritmo)
    print(f"{algoritmo.__class__.__name__} → treinado ✔")

In [ ]:
resultados_exp2 = []

for modelo in listaModelos_exp2:
    y_pred = modelo.predict(X_test)
    acc  = accuracy_score(y_test, y_pred)
    f1   = f1_score(y_test, y_pred)
    rec  = recall_score(y_test, y_pred)
    resultados_exp2.append({'modelo': modelo.__class__.__name__, 'acc': acc, 'f1': f1, 'recall': rec})

    print(f"\n{'='*50}")
    print(f"  {modelo.__class__.__name__} (Exp 2)")
    print(f"{'='*50}")
    print(classification_report(y_test, y_pred, target_names=['Adimplente', 'Inadimplente']))

    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Adimplente','Inadimplente'],
                yticklabels=['Adimplente','Inadimplente'])
    plt.title(f'Matriz de Confusão — {modelo.__class__.__name__} (Exp 2)')
    plt.ylabel('Real'); plt.xlabel('Previsto')
    plt.tight_layout(); plt.show()

## Experimento 3 — VotingClassifier (Ensemble)

In [ ]:
print("EXPERIMENTO 3 - VotingClassifier")

voting = VotingClassifier(
    estimators=[
        ('lr', listaModelos_exp2[1]),  # Pipeline(LogisticRegression)
        ('rf', listaModelos_exp2[0]),  # RandomForest
        ('gb', listaModelos_exp2[3])   # GradientBoosting
    ],
    voting='soft'
)

voting.fit(X_train_bal, y_train_bal)
y_pred_voting = voting.predict(X_test)

print(f"\n{'='*50}")
print(f"  VotingClassifier (Exp 3)")
print(f"{'='*50}")
print(classification_report(y_test, y_pred_voting, target_names=['Adimplente', 'Inadimplente']))

cm = confusion_matrix(y_test, y_pred_voting)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Adimplente','Inadimplente'],
            yticklabels=['Adimplente','Inadimplente'])
plt.title('Matriz de Confusão — VotingClassifier (Exp 3)')
plt.ylabel('Real'); plt.xlabel('Previsto')
plt.tight_layout(); plt.show()

acc_voting = accuracy_score(y_test, y_pred_voting)
f1_voting  = f1_score(y_test, y_pred_voting)
rec_voting = recall_score(y_test, y_pred_voting)
print(f"Acurácia: {acc_voting:.4f} | F1: {f1_voting:.4f} | Recall: {rec_voting:.4f}")

## Experimento 4 — LogisticRegression com Ajuste de Threshold

Com dados desbalanceados, o threshold padrão (0.5) não é necessariamente o ideal. Testamos diferentes pontos de corte para maximizar o F1 da classe inadimplente.

In [ ]:
print("EXPERIMENTO 4 - Ajuste de Threshold na LogisticRegression")

# Melhor modelo do Exp 2 (LogisticRegression via Pipeline)
modelo_lr = listaModelos_exp2[1]
y_proba = modelo_lr.predict_proba(X_test)[:, 1]

# Testar diferentes thresholds
resultados_threshold = []
for threshold in np.arange(0.25, 0.55, 0.05):
    y_pred_t = (y_proba >= threshold).astype(int)
    f1  = f1_score(y_test, y_pred_t)
    rec = recall_score(y_test, y_pred_t)
    acc = accuracy_score(y_test, y_pred_t)
    resultados_threshold.append({'threshold': threshold, 'f1': f1, 'recall': rec, 'acc': acc})
    print(f"Threshold {threshold:.2f}: F1={f1:.4f} | Recall={rec:.4f} | Acurácia={acc:.4f}")

df_thresh = pd.DataFrame(resultados_threshold)
melhor = df_thresh.loc[df_thresh['f1'].idxmax()]
print(f"\nMelhor threshold: {melhor['threshold']:.2f} → F1={melhor['f1']:.4f}")

In [ ]:
# Visualizar F1 x Threshold
plt.figure(figsize=(8, 4))
plt.plot(df_thresh['threshold'], df_thresh['f1'],    marker='o', label='F1 Inadimplente')
plt.plot(df_thresh['threshold'], df_thresh['recall'], marker='s', label='Recall')
plt.plot(df_thresh['threshold'], df_thresh['acc'],    marker='^', label='Acurácia')
plt.axvline(x=melhor['threshold'], color='red', linestyle='--', label=f"Melhor threshold ({melhor['threshold']:.2f})")
plt.xlabel('Threshold')
plt.ylabel('Score')
plt.title('Métricas por Threshold — LogisticRegression')
plt.legend(); plt.tight_layout(); plt.show()

## Curva ROC — Comparação dos Modelos

In [ ]:
plt.figure(figsize=(9, 6))

for nome, modelo in [
    ('RandomForest (Exp2)',       listaModelos_exp2[0]),
    ('LogisticRegression (Exp2)', listaModelos_exp2[1]),
    ('GaussianNB (Exp2)',         listaModelos_exp2[2]),
    ('GradientBoosting (Exp2)',   listaModelos_exp2[3]),
    ('VotingClassifier (Exp3)',   voting)
]:
    proba = modelo.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc = roc_auc_score(y_test, proba)
    plt.plot(fpr, tpr, label=f"{nome} (AUC={auc:.3f})")

plt.plot([0, 1], [0, 1], 'k--', label='Aleatório')
plt.xlabel('FPR (Falso Positivo)')
plt.ylabel('TPR (Recall)')
plt.title('Curva ROC — Comparação dos Modelos')
plt.legend(loc='lower right')
plt.tight_layout(); plt.show()

## Comparação Final dos Experimentos

In [ ]:
print(f"{'='*70}")
print(f"{'Experimento':<40} {'Acurácia':<12} {'F1 Inad.':<12} {'Recall Inad.'}")
print(f"{'='*70}")

for r in resultados_exp1:
    print(f"Exp1 {r['modelo']:<35} {r['acc']:.4f}       {r['f1']:.4f}       {r['recall']:.4f}")

print(f"{'-'*70}")

for r in resultados_exp2:
    print(f"Exp2 {r['modelo']:<35} {r['acc']:.4f}       {r['f1']:.4f}       {r['recall']:.4f}")

print(f"{'-'*70}")
print(f"Exp3 {'VotingClassifier':<35} {acc_voting:.4f}       {f1_voting:.4f}       {rec_voting:.4f}")

print(f"{'-'*70}")
# Melhor threshold para LR
best_thresh = melhor['threshold']
y_pred_best = (y_proba >= best_thresh).astype(int)
f1_best  = f1_score(y_test, y_pred_best)
rec_best = recall_score(y_test, y_pred_best)
acc_best = accuracy_score(y_test, y_pred_best)
print(f"Exp4 LogReg threshold={best_thresh:.2f}               {acc_best:.4f}       {f1_best:.4f}       {rec_best:.4f}")

print(f"{'='*70}")

In [ ]:
# Salvar melhor modelo (LogisticRegression com threshold ajustado)
NOMEMODELO = 'modelo_inadimplencia_v1.pickle'

payload = {
    'modelo': listaModelos_exp2[1],
    'threshold': float(best_thresh)
}

with open(NOMEMODELO, 'wb') as f:
    pickle.dump(payload, f)
print(f"Modelo salvo como '{NOMEMODELO}' ✔")

# Validação no conjunto de holdout
with open(NOMEMODELO, 'rb') as f:
    carregado = pickle.load(f)

modelo_val   = carregado['modelo']
threshold_val = carregado['threshold']

y_proba_val = modelo_val.predict_proba(X_val)[:, 1]
y_pred_val  = (y_proba_val >= threshold_val).astype(int)

print(f"\nAcurácia na validação: {accuracy_score(y_val, y_pred_val):.4f}")
print(classification_report(y_val, y_pred_val, target_names=['Adimplente', 'Inadimplente']))

In [ ]:
# Exemplo de predição em produção
print("Exemplo de predição em produção:")
novo_cliente = pd.DataFrame([X_val.iloc[0]], columns=X_val.columns)
proba_cliente = modelo_val.predict_proba(novo_cliente)[0][1]
pred = 1 if proba_cliente >= threshold_val else 0
print(f"Probabilidade de inadimplência: {proba_cliente:.2%}")
print(f"Predição (threshold={threshold_val:.2f}): {'Inadimplente ⚠️' if pred == 1 else 'Adimplente ✔'}")

# Conclusão Final do Projeto

## Dataset
- **Fonte:** Lending Club (2007–2018)
- **Registros:** ~115 mil (amostra de 200k)
- **Features:** 7 (após remoção de `parcela` por multicolinearidade)
- **Target:** `inadimplente` (0 = Adimplente, 1 = Inadimplente)

---

## Experimentos Realizados

| Experimento | Descrição | Melhor Modelo |
|------------|----------|---------------|
| Exp 1 | Sem SMOTE (baseline) | GaussianNB |
| Exp 2 | Com SMOTE | LogisticRegression |
| Exp 3 | VotingClassifier (ensemble) | Ensemble |
| Exp 4 | Threshold ajustado | LogisticRegression |

---

## Principais Conclusões

- **SMOTE foi decisivo** para modelos tradicionais: Recall aumentou de ~0.07 para ~0.62
- **Remover `parcela`** (correlação 0.95 com valor_emprestimo) não impactou os resultados significativamente
- **Ajuste de threshold** permite balancear precision/recall conforme o custo do negócio
- **VotingClassifier não superou** modelos individuais

## Interpretação de Negócio

Em problemas de crédito, os erros têm **custos assimétricos**:
- ⚠️ **Falso Negativo** (inadimplente classificado como adimplente) → custo alto (perda financeira)
- **Falso Positivo** (adimplente classificado como inadimplente) → custo menor (oportunidade perdida)

Por isso, o modelo foi configurado priorizando o **Recall da classe inadimplente**, utilizando um threshold ajustado abaixo de 0.5 para reduzir o risco de concessão de crédito inadequada.